In [21]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import holidays
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from xgboost import XGBRegressor


In [22]:
df = pd.read_csv('dados/station_data_dataverse.csv')
df.columns = df.columns.str.lower()
df = df.drop_duplicates()

In [23]:
df['created'] =pd.to_datetime(df['created'], errors='coerce')
df = df.dropna(subset=['created', 'kwhtotal'])
df= df[df['kwhtotal'] > 0]
df.shape

(3340, 24)

In [24]:
serie = df.set_index('created')['kwhtotal'].resample('D').sum()
serie = serie.asfreq('D', fill_value=0)
serie.shape

(321,)

In [25]:
print(serie.index.min(), serie.index.max())
print((serie == 0).sum(), 'dias sem sessão')

dias_zero = serie[serie == 0]
print(dias_zero.index.dayofweek.value_counts().sort_index())

0014-11-18 00:00:00 0015-10-04 00:00:00
84 dias sem sessão
created
0     8
1     5
2     4
3     6
4     9
5    20
6    32
Name: count, dtype: int64


In [26]:
feriados = holidays.US()

dados = pd.DataFrame({'y': serie})
dados['dia_semana'] = dados.index.dayofweek
dados['mes'] = dados.index.month
dados['feriado'] = [int(d in feriados) for d in dados.index]
dados.head()

,y,dia_semana,mes,feriado
created,,,,
14-11-18,13.39,1,11,0
14-11-19,18.77,2,11,0
14-11-20,6.95,3,11,0
14-11-21,14.14,4,11,0
14-11-22,0.00,5,11,0


In [27]:
corte = int(len(dados) * 0.8)
treino, teste = dados.iloc[:corte], dados.iloc[corte:]
len(treino), len(teste)

(256, 65)

In [28]:
def avaliar(real, previsto):
    mae = mean_absolute_error(real, previsto)
    rmse = np.sqrt(mean_squared_error(real, previsto))
    return round(mae, 2), round(rmse, 2)

resultados, previsoes = {}, {}

previsoes['baseline'] = np.full(len(teste), treino['y'].mean())
resultados['baseline'] = avaliar(teste['y'], previsoes['baseline'])

In [29]:
melhor_aic= None
for ordem in [(1, 1, 1), (2, 1, 2), (7, 1, 1)]:
    ajuste = ARIMA(treino['y'], order=ordem).fit()
    if melhor_aic is None or ajuste.aic < melhor_aic:
        melhor_aic, melhor_ordem, arima_modelo = ajuste.aic, ordem, ajuste
print('ordem escolhida:', melhor_ordem)

previsoes['ARIMA'] = arima_modelo.forecast(steps=len(teste)).values
resultados['ARIMA'] = avaliar(teste['y'], previsoes['ARIMA'])

ordem escolhida: (7, 1, 1)


In [30]:
df_p = treino.reset_index().rename(columns={'created': 'ds'})[['ds', 'y']]

prophet_modelo = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False)
prophet_modelo.add_country_holidays(country_name='US')
prophet_modelo.fit(df_p)

futuro = pd.DataFrame({'ds': teste.index})
previsoes['Prophet'] = prophet_modelo.predict(futuro)['yhat'].values
resultados['Prophet'] = avaliar(teste['y'], previsoes['Prophet'])

10:53:09 - cmdstanpy - INFO - Chain [1] start processing
10:53:09 - cmdstanpy - INFO - Chain [1] done processing


In [32]:
colunas_x = ['dia_semana', 'mes', 'feriado']

xgb_modelo = XGBRegressor(n_estiamtors=200, max_depth=3, learning_rate=0.1, random_state=42)
xgb_modelo.fit(treino[colunas_x], treino['y'])

previsoes['XGBoost'] = xgb_modelo.predict(teste[colunas_x])
resultados['XGBoost'] = avaliar(teste['y'], previsoes['XGBoost'])
